# OfflinePreprocessor — Manual Validation Notebook

Runs the pipeline stage by stage on real data.  
Each cell ends with assertions; a passing cell means that stage is behaving correctly.

**This notebook is for validation only — not part of the app.**

In [ ]:
import sys
from pathlib import Path

# ── Point these at your data ──────────────────────────────────────────────
DATA_DIR    = Path("../../data/raw/sub_001/")   # directory that contains the .vhdr file
CONFIG_PATH = Path("../experiment_config.yaml")
# ─────────────────────────────────────────────────────────────────────────

# Make src/ importable without installing the package
sys.path.insert(0, str(Path("../src").resolve()))

import mne
import numpy as np
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

from backend.core.settings_manager import SettingsManager
from backend.offline_phase.preprocessor import OfflinePreprocessor

mne.set_log_level("WARNING")
print("Imports OK")

---
## Config & Preprocessor Init

In [ ]:
settings = SettingsManager(CONFIG_PATH)
preprocessing_params = settings.get_preprocessing_params()
event_mapping_str_to_int = settings.get_event_mapping()   # {name: trigger_id} — MNE format

# Find the .vhdr file (mirrors OfflineOrchestrator._find_vhdr())
_vhdrs = list(DATA_DIR.glob("*.vhdr"))
assert _vhdrs, f"No .vhdr file found in {DATA_DIR}"
vhdr = _vhdrs[0]
if len(_vhdrs) > 1:
    print(f"⚠ Multiple .vhdr files found; using {vhdr.name}")

preprocessor = OfflinePreprocessor(
    data_dir=DATA_DIR,
    preprocessing_settings=preprocessing_params,
)

print(f"Subject ID : {preprocessor.subject_id}")
print(f"Data dir   : {preprocessor.data_dir}")
print(f"VHDR file  : {vhdr}")
print(f"Event map  : {event_mapping_str_to_int}")

# ── Assertions ────────────────────────────────────────────────────────────
assert vhdr.exists(),                      "VHDR path does not exist on disk"
assert len(event_mapping_str_to_int) > 0,  "Event mapping is empty — check config"
print("\n✓ Config & init OK")

---
## Stage 1 — Load Raw

In [ ]:
# ── WORKAROUND for recordings that exceed available RAM ───────────────────
# Loading with preload=True can raise MemoryError on large files.
# If that happens, set TMAX to a duration in seconds to crop before loading.
# TODO: implement proper large-file support (memmap) in OfflineOrchestrator.
TMAX = 300   # seconds — None = full file; e.g. 3600 for first hour only
# ─────────────────────────────────────────────────────────────────────────

# Mirrors OfflineOrchestrator._load_eeg_raw(): load BrainVision file,
# apply standard 10-20 montage, keep only EEG/EOG/ECG channels.
_raw = mne.io.read_raw_brainvision(vhdr, preload=(TMAX is None), verbose=False)
_montage = mne.channels.make_standard_montage("standard_1020")
_raw.set_montage(_montage, match_case=False, on_missing="warn")

_montage_names = {ch.lower() for ch in _montage.ch_names}
_eeg_picks = [i for i in mne.pick_types(_raw.info, eeg=True)
              if _raw.ch_names[i].lower() in _montage_names]
_eog_picks = mne.pick_types(_raw.info, eog=True).tolist()
_ecg_picks = mne.pick_types(_raw.info, ecg=True).tolist()
_keep = sorted(set(_eeg_picks) | set(_eog_picks) | set(_ecg_picks))
_dropped = [_raw.ch_names[i] for i in range(len(_raw.ch_names)) if i not in _keep]
if _dropped:
    print(f"Dropping non-EEG/EOG/ECG channels: {_dropped}")
_raw.pick(_keep)

if TMAX is not None:
    _raw.crop(tmax=TMAX)
    _raw.load_data(verbose=False)

preprocessor.raw = _raw
raw = preprocessor.raw

print(f"Channels   : {len(raw.ch_names)}  ({len(mne.pick_types(raw.info, eeg=True))} EEG)")
print(f"Sampling Hz: {raw.info['sfreq']}")
print(f"Duration s : {raw.times[-1]:.1f}")
print(f"Channel types: {set(raw.get_channel_types())}")

# ── Assertions ────────────────────────────────────────────────────────────
assert len(raw.ch_names) > 0, "No channels loaded"
assert raw.info['sfreq'] > 0, "Invalid sampling rate"
assert raw.times[-1] > 10,   "Recording suspiciously short (< 10 s) — check file"
assert raw.preload,           "Signal data should be in RAM"
print("\n✓ Raw loaded OK")

In [ ]:
# [Optional] Inspect all annotations in the recording and compare to config
# BrainVision annotation descriptions may differ from config event names.
# Use this to decide whether event_mapping_str_to_int needs overriding below.

_events, _found_id = mne.events_from_annotations(raw, verbose=False)

print("Annotations found in recording:")
for desc, code in sorted(_found_id.items(), key=lambda x: x[1]):
    count = (_events[:, 2] == code).sum()
    print(f"  code {code:4d}  '{desc}'  —  {count} occurrences")

print(f"\nConfig event mapping (name → code): {event_mapping_str_to_int}")

_config_codes = set(event_mapping_str_to_int.values())
_found_codes  = set(_found_id.values())
_matched = _config_codes & _found_codes
_unmatched = _config_codes - _found_codes
if _unmatched:
    print(f"\n⚠  Config codes not found in recording: {_unmatched}")
    print("   → Override event_mapping_str_to_int in the cell before Stage 4.")
else:
    print(f"\n✓ All config codes found in recording ({_matched})")

In [ ]:
# [Optional] Plot raw PSD to inspect the unfiltered spectrum
spectrum = raw.compute_psd(picks="eeg", fmax=200)
spectrum.plot()
plt.suptitle("Raw PSD (before filtering)")
plt.tight_layout()

---
## Stage 2a — Filter

In [ ]:
bp = preprocessing_params["bandpass"]
print(f"Applying bandpass {bp['l_freq']}–{bp['h_freq']} Hz ({bp['method']}), notch: {bp.get('notch')} Hz")

preprocessor._filter()


# ── Assertions ────────────────────────────────────────────────────────────
print("\n✓ Filter Completed")

In [ ]:
# [Optional] Compare PSD before/after to confirm bandpass is applied
spectrum_filtered = raw.compute_psd(picks="eeg", fmax=200)
spectrum_filtered.plot()
plt.suptitle("PSD after bandpass filter")
plt.tight_layout()

---
## Stage 2b — Resample

In [ ]:
target = preprocessing_params['resample']['target_rate']
print(f"Before — sfreq: {raw.info['sfreq']:.0f} Hz, {raw.n_times} samples")

preprocessor._resample()

print(f"After  — sfreq: {raw.info['sfreq']:.0f} Hz, {raw.n_times} samples  (target: {target} Hz)")

# ── Assertions ────────────────────────────────────────────────────────────
assert raw.info['sfreq'] == target, f"Expected sfreq {target}, got {raw.info['sfreq']}"
print("\n✓ Resample OK")

---
## Stage 2c — Bad Channel Detection

In [ ]:
stds_before = raw.get_data(picks="eeg").std(axis=1)
print(f"Before — std range: {stds_before.min():.2e} – {stds_before.max():.2e} V  ({len(stds_before)} EEG channels)")

preprocessor._detect_bad_channels()

bads = preprocessor._bad_channels
stds_after = raw.get_data(picks="eeg").std(axis=1)
print(f"After  — std range: {stds_after.min():.2e} – {stds_after.max():.2e} V")
print(f"Detected and interpolated ({len(bads)}): {bads}")
print(f"raw.info['bads'] after reset: {raw.info['bads']}")

# ── Assertions ────────────────────────────────────────────────────────────
n_eeg = len(mne.pick_types(raw.info, eeg=True))
assert len(bads) < n_eeg * 0.3, f"Too many bad channels ({len(bads)}/{n_eeg}) — check data quality"
assert raw.info['bads'] == [],  "raw.info['bads'] should be empty after interpolation"
print("\n✓ Bad channel detection OK")

In [ ]:
# [Optional] Plot channel stds to visually confirm outliers were caught
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(stds_after)), stds_after)
ax.set_xlabel("Channel index (EEG)")
ax.set_ylabel("Std (V)")
ax.set_title("Channel standard deviations after bad-channel interpolation")
plt.tight_layout()

---
## Stage 2d — Average Reference

In [ ]:
mean_before = np.abs(raw.get_data(picks="eeg").mean(axis=0)).max()
print(f"Before — max |channel mean|: {mean_before:.2e} V")

preprocessor._reference()

max_mean = np.abs(raw.get_data(picks="eeg").mean(axis=0)).max()
print(f"After  — max |channel mean|: {max_mean:.2e} V  (expect ~0)")

# ── Assertions ────────────────────────────────────────────────────────────
assert max_mean < 1e-13, f"Average reference failed — mean is {max_mean:.2e}"
print("\n✓ Reference OK")

---
## Stage 3 — ICA Fit

This can take 1–2 minutes on real data.

In [ ]:
ica_cfg = preprocessing_params['ica']
print(f"Before — {len(raw.ch_names)} channels, fitting {ica_cfg['n_components']} components (method: {ica_cfg['method']})")

suggested = preprocessor._fit_ica()

print(f"After  — {preprocessor.ica.n_components_} components fitted")
print(f"Suggested artifact components: {suggested}")

# ── Assertions ────────────────────────────────────────────────────────────
assert preprocessor.ica is not None,                        "ICA object not stored"
assert isinstance(suggested, list),                         "suggest must be a list"
assert all(isinstance(i, int) for i in suggested),          "suggested indices must be ints"
assert preprocessor.ica.exclude == [],                      "_fit_ica must NOT set exclude (that's the UI's job)"
print("\n✓ ICA fit OK")

In [ ]:
# [Optional] All component topomaps — scan for spatial artifact patterns
# Eye artifacts : frontal distribution (Fp1/Fp2 region)
# Muscle artifacts: peripheral electrodes, irregular shape
# Cardiac        : widespread, often right-temporal
preprocessor.ica.plot_components(picks=range(preprocessor.ica.n_components_))

In [ ]:
# [Optional] All component time series — identify blink/heartbeat/muscle patterns
# Eye blinks    : large slow deflections, strongest in frontal components
# Heartbeat (ECG): regular ~1 Hz rhythm
# Muscle        : high-frequency bursts
preprocessor.ica.plot_sources(preprocessor.raw)

In [ ]:
# [Optional] Detailed per-component view: topomap + PSD + time series
# Set INSPECT to the component indices you want to examine closely.
# Start with suggested; add any that looked suspicious in the plots above.
INSPECT = suggested if suggested else list(range(min(5, preprocessor.ica.n_components_)))
for i in INSPECT:
    preprocessor.ica.plot_properties(preprocessor.raw, picks=i)

---
## Stage 4 — ICA Apply + Epoch

Review the plots above and identify artifact components, then set `EXCLUDE_COMPONENTS`.

**What to look for:**
- **Eye blinks**: frontal topomap (Fp1/Fp2), large slow deflections in time series
- **Lateral eye movements**: bipolar frontal topomap (left/right asymmetry)
- **Heartbeat**: widespread topomap, regular ~1 Hz pulses in time series
- **Muscle**: peripheral topomap, high-frequency bursts in time series

**Workflow:**
1. `suggested` is pre-populated from EOG/ECG auto-detection (empty if no physio channels)
2. Look at the plots above and note any suspicious component indices
3. Add them to `EXCLUDE_COMPONENTS` in the cell below, then run it

In [ ]:
# ── Event mapping override (optional) ─────────────────────────────────────
# Run the event inspection cell above first (Stage 1 → Inspect Annotations).
# If the recording's annotation descriptions don't match the config names,
# set the mapping here using the CODES found in the recording printout.
#
# Example — recording uses numeric string keys instead of colour names:
#   event_mapping_str_to_int = {"S  1": 1, "S  2": 2, "S  3": 3}
#
# Leave unchanged to use whatever the config defined.
event_mapping_str_to_int = { "first": 100, "second": 101, "third": 1 , "fourth": 31, "fifth": 32, "sixth": 33, "seventh": 41, "eighth": 45, "ninth": 51, "tenth": 52, "eleventh": 61, "twelfth": 62, "thirteenth": 10001, "fourteenth": 10002 }   # ← uncomment and edit if needed

print(f"Using event mapping: {event_mapping_str_to_int}")

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────
# Start from auto-suggestions (may be empty if no EOG/ECG channels):
EXCLUDE_COMPONENTS = list(suggested)

# After visual inspection, add any components you identified as artifacts.
# Example (replace with your own findings from the plots above):
#   IC00 — frontal topomap + large slow deflections in sources → eye blink
#   IC03 — right-temporal, regular ~1 Hz pulses               → heartbeat
EXCLUDE_COMPONENTS += [0, 3]
# ─────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("../data_vault") / preprocessor.subject_id

print(f"Before — excluding ICA components: {EXCLUDE_COMPONENTS}")
print(f"         tmin={preprocessing_params['epochs']['tmin']} s, tmax={preprocessing_params['epochs']['tmax']} s")

preprocessor.run_step2_finish_pipeline(
    exclude_components=EXCLUDE_COMPONENTS,
    event_mapping=event_mapping_str_to_int,
    output_dir=OUTPUT_DIR,
)

epochs = preprocessor.epochs
event_counts = {k: int((epochs.events[:, 2] == v).sum()) for k, v in epochs.event_id.items()}
print(f"After  — epochs shape: {epochs.get_data().shape}  (n_epochs, n_ch, n_times)")
print(f"         event counts: {event_counts}")

# ── Assertions ────────────────────────────────────────────────────────────
assert epochs is not None,          "Epochs not created"
assert len(epochs) > 0,             "Zero epochs — no matching triggers found"
assert epochs.get_data().ndim == 3, "Epochs must be 3D (n_epochs, n_ch, n_times)"

ep_params = preprocessing_params['epochs']
assert np.isclose(epochs.tmin, ep_params['tmin'], atol=1/epochs.info['sfreq']), "tmin mismatch"
assert np.isclose(epochs.tmax, ep_params['tmax'], atol=1/epochs.info['sfreq']), "tmax mismatch"
print("\n✓ ICA apply + epoching OK")

In [ ]:
# [Optional] Plot the ERP to sanity-check the epoch quality
evoked = epochs.average()
evoked.plot()
plt.suptitle("Grand average ERP (all conditions)")
plt.tight_layout()

In [ ]:
# [Optional] Per-condition ERPs
for cond, event_id in epochs.event_id.items():
    n = (epochs.events[:, 2] == event_id).sum()
    print(f"  {cond:20s}: {n} epochs")
    epochs[cond].average().plot(titles=cond)

---
## Stage 5 — Save & Online State

In [ ]:
# Check the .fif was saved
saved_files = list(OUTPUT_DIR.glob("*.fif"))
print(f"Saved files: {[f.name for f in saved_files]}")

# ── Assertions ────────────────────────────────────────────────────────────
assert len(saved_files) == 1,                               "Expected exactly one .fif file"
assert preprocessor.subject_id in saved_files[0].name,     "Subject ID missing from filename"
print("\n✓ Save OK")

In [ ]:
online_state = preprocessor.export_online_state()

print("online_state keys and types:")
for k, v in online_state.items():
    if hasattr(v, 'shape'):
        print(f"  {k:25s}: np.ndarray  shape={v.shape}")
    else:
        print(f"  {k:25s}: {type(v).__name__}  = {v}")

# ── Assertions ────────────────────────────────────────────────────────────
required_keys = {"bad_channels", "ica_unmixing", "ica_mixing",
                 "ica_pca_components", "ica_pca_mean", "ica_exclude",
                 "ch_names", "sfreq_offline"}
assert required_keys <= online_state.keys(), f"Missing keys: {required_keys - online_state.keys()}"

n_comp = preprocessor.ica.n_components_
n_ch   = len(online_state["ch_names"])  # EEG channels only — matches pca_components
assert online_state["ica_unmixing"].shape       == (n_comp, n_comp), "unmixing shape wrong"
assert online_state["ica_mixing"].shape         == (n_comp, n_comp), "mixing shape wrong"
assert online_state["ica_pca_components"].shape == (n_comp, n_ch),   "pca_components shape wrong"
assert online_state["ica_exclude"]              == EXCLUDE_COMPONENTS, "exclude mismatch"
assert online_state["sfreq_offline"]            == preprocessing_params['resample']['target_rate']
assert isinstance(online_state["ch_names"], list)
print("\n✓ export_online_state OK")

---
## Summary

If all cells passed, the full offline preprocessing pipeline is working correctly on real data.

**Next steps:**
- Pass `online_state` to `ModelEvaluator` for cross-validation
- Pass `online_state` to `ModelTrainer` to produce the final `decoder_pipeline.joblib`